# Portfolio Analysis

**Positions truth:** broker exports (Fidelity now, Robinhood later) ·
**Prices/history:** yfinance · **Classification:** `config/symbol_map.yaml`

This notebook reads the Parquet produced by the ingest pipeline. To refresh data,
run `uv run invest-ingest` in a terminal (or the cell below), then re-run from the top.

In [ ]:
# Optional: refresh the data store from the latest broker export + yfinance.
# Comment out to just read existing Parquet.
from invest import pipeline
_ = pipeline.run(history_period="3y", use_network=True, verbose=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from invest import config, analysis

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

positions = pd.read_parquet(config.POSITIONS_PARQUET)
try:
    history = pd.read_parquet(config.PRICES_PARQUET)
except FileNotFoundError:
    history = pd.DataFrame()

total = positions["market_value"].sum()
print(f"Total portfolio value: ${total:,.2f}")
print(f"Holdings: {len(positions)} across {positions['account_name'].nunique()} accounts "
      f"and {positions['broker'].nunique()} broker(s)")

## Positions — how each price was resolved

In [ ]:
positions[[
    "broker", "account_name", "symbol", "description", "asset_class",
    "price_source", "price_used", "quantity", "price", "market_value",
]].sort_values("market_value", ascending=False)

### Sanity check: anything unclassified?
New holdings the symbol map doesn't cover yet fall back to `fidelity_csv` pricing
and `unclassified`. Add them to `config/symbol_map.yaml`.

In [ ]:
unknown = positions[positions["asset_class"] == "unclassified"]
if len(unknown):
    display(unknown[["broker", "symbol", "description", "market_value"]])
else:
    print("All holdings classified ✓")

## Allocation by asset class

In [ ]:
alloc = analysis.allocation_by(positions, "asset_class")
display(alloc)

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(alloc["market_value"], labels=alloc.index,
       autopct=lambda p: f"{p:.1f}%", startangle=90, counterclock=False)
ax.set_title("Allocation by asset class")
plt.show()

## Allocation by account

In [ ]:
acct = analysis.account_summary(positions)
display(acct)

ax = acct["market_value"].plot(kind="barh", figsize=(9, 4), title="Market value by account")
ax.set_xlabel("Market value ($)")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

## Top holdings & concentration

In [ ]:
display(analysis.top_holdings(positions, 10))

c = analysis.concentration(positions)
print(f"HHI (Herfindahl):       {c['hhi']:.3f}   (1.0 = single holding)")
print(f"Effective # of holdings: {c['effective_holdings']:.1f}")
print(f"Top-1 weight:            {c['top1_weight']:.1%}")
print(f"Top-5 weight:            {c['top5_weight']:.1%}")

## Portfolio value over time (all holdings)

Holds **today's** share counts flat across the window and values every holding by
the best source available — actual yfinance history where we have it, a proxy
index for index-tracking funds (shape only), and a flat current value for cash and
proprietary 401k pools with no public quote. The stacked total meets your current
portfolio value at the right edge.

**Risk/composition lens, not performance** — it ignores when you actually bought,
and "flat" holdings show no movement. The coverage line says how much of the path
is real vs. approximated.

In [ ]:
pvh = analysis.portfolio_value_history(positions, history)  # date x asset_class
if pvh.empty:
    print("No price history available — run the pipeline with network enabled.")
else:
    total = pvh.sum(axis=1)

    # Stacked area over every asset class -> the total IS portfolio value.
    ax = pvh.plot.area(figsize=(11, 5), linewidth=0)
    ax.set_title("Portfolio value over time — current holdings, all assets")
    ax.set_ylabel("Value ($)")
    ax.legend(loc="upper left", fontsize=8, ncol=2)
    plt.tight_layout(); plt.show()

    dd = (total / total.cummax() - 1).min()
    print(f"Latest total: ${total.iloc[-1]:,.0f}   |   max drawdown of total: {dd:.1%}")

    # How much of that value path is real vs. approximated.
    tr = analysis.holding_value_treatment(positions, history)
    cov = tr.groupby("treatment")["market_value"].sum()
    pct = (cov / cov.sum() * 100).round(1)
    print("\nPrice-history coverage of current value:")
    for t in ["live", "proxy", "flat"]:
        if t in cov.index:
            print(f"  {t:5}: ${cov[t]:>12,.0f}  ({pct[t]:>5}%)")
    print("  live = actual history · proxy = index stand-in (shape only) · "
          "flat = held at current value (no public history)")

## Benchmark & holding risk summary

In [ ]:
summary = analysis.risk_summary(history)
if summary.empty:
    print("No history loaded.")
else:
    display(summary.sort_values("annual_return", ascending=False)
                   .style.format({"annual_return": "{:.1%}", "annual_vol": "{:.1%}",
                                  "max_drawdown": "{:.1%}", "sharpe_naive": "{:.2f}"}))

---
*yfinance is an unofficial Yahoo data source — treat prices as research-grade.
No tax-lot or transaction data is modeled here; positions only.*